# 🤖 Agentic RAG Workshop — Complete in Just 4 Hours

**Agentic RAG: From Zero to Hero** | 4-hour Workshop

---

### 🎯 Learning Objectives

By the end of this workshop, learners will be able to:
1. **Prepare data for RAG** — Chunking + Embedding + VectorDB
2. **Build AI Agents** — Use Google ADK to create Agents + Custom Tools
3. **Build Agentic RAG** ⭐ — An Agent that searches from VectorDB + reasons + answers questions on its own
4. **Measure quality** — Use LLM-as-Judge to evaluate the Agent

### ⏱️ Timeline

| Part | Time | Content |
|------|:----:|--------|
| 📢 Part 1 | 1 hr 20 min | Data → Chunk → Embed → Qdrant → Retrieve |
| ☕ Break | 10 min | |
| 📢 Part 2 | 1 hr 30 min | Agent → Tool → RAG Agent → Agentic RAG |
| 🧪 Part 3 | 1 hr | Exercises + Q&A |

---
## 📢 Part 1: Data → VectorDB

### What is RAG?

**R**etrieval-**A**ugmented **G**eneration = Search + Generate

```
LLMs are powerful, but they don't know our data
→ So we need to "retrieve" relevant information first
→ Then the LLM will "generate" an answer from that information
```

| Today's Pipeline |
|---|
| 📄 Text → ✂️ Chunking → 🔢 Embedding → 🗄️ Qdrant → 🔎 Retrieve → 🤖 Agent → 💬 Answer |

## 📦 Section 0: Setup

Install all required libraries

In [ ]:
%%time
# Install libraries
!pip install -q google-adk \
    google-genai \
    sentence-transformers \
    qdrant-client \
    langchain-text-splitters \
    scikit-learn

print('✅ Installation complete!')
print('⚠️ After the first install, please restart the runtime: Runtime → Restart session → then run again starting from the next cell')

In [ ]:
%%time
# Set Gemini API Key
import os

try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('✅ Loaded API Key from Colab Secrets')
except Exception:
    os.environ['GOOGLE_API_KEY'] = input('🔑 Please paste your Gemini API Key: ')

# Test API
from google import genai
client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
resp = client.models.generate_content(model='gemini-2.5-flash', contents='Hello, please answer briefly in 1 sentence')
print(f'🤖 Gemini: {resp.text}')

### 📄 Prepare Sample Data

Create Thai case study sample data for use throughout the workshop

In [ ]:
%%time
# Create sample data
import os
os.makedirs('data', exist_ok=True)

sample_texts = {
    'case_ict_mahidol.txt': '''Case Study: AI at the Faculty of ICT, Mahidol University

The Faculty of Information and Communication Technology (ICT), Mahidol University,
is home to the Mahidol AI Center, Mahidol's artificial intelligence institute.
It has the MIKE (Machine Intelligence and Knowledge Engineering) research cluster.

Its research focuses on AI in medicine, genomics, and federated learning.
Notable projects include Siribot, a chatbot for Siriraj Hospital,
AI Medical Imaging for analyzing X-ray images, and VIS4ION, a navigation system for the visually impaired.

The ICT faculty also uses RAG to build an AI Tutor system
that helps answer course questions from teaching materials.
Students can ask questions 24/7, with answers retrieved from lecture notes, slides, and textbooks.''',

    'case_healthcare.txt': '''Case Study: AI in Thai Healthcare

Siriraj Hospital uses AI to analyze medical images,
such as detecting lung cancer from X-ray images with deep learning.
Test results showed that the AI achieved 95% accuracy.

NLP is also used to analyze electronic medical records,
reducing record-reading time from 15 minutes to 2 minutes per case.

Challenge: Thai medical language data is limited in quantity,
so transfer learning from English models is required.''',

    'case_banking.txt': '''Case Study: AI in Banking and Finance

Kasikornbank developed a chatbot system that uses LLM together with RAG
to answer customer questions about financial products.

The RAG system retrieves information from an internal knowledge base,
including product documents, service conditions, and FAQs.
The LLM generates natural-language answers from the retrieved information.

Results: reduced call center workload by 40%,
customer satisfaction increased by 25%,
and a vector database was used to store embeddings + hybrid search.''',

    'case_agriculture.txt': '''Case Study: AI in Smart Agriculture

Smart Farming in Thailand uses AI to analyze satellite images
and data from IoT sensors to forecast yields and monitor crop diseases.

A computer vision system detects rice diseases from images of rice leaves,
using a CNN trained on more than 50,000 rice disease images.

NLP is also used to analyze agricultural price information
from news, government reports, and social media,
helping farmers make decisions about planting and selling.'''
}

for fname, content in sample_texts.items():
    with open(f'data/{fname}', 'w', encoding='utf-8') as f:
        f.write(content)

print(f'✅ Created {len(sample_texts)} sample files')
for fname in sorted(sample_texts.keys()):
    print(f'  📄 {fname}')

---
## ✂️ Section 1: Chunking — Split Text into Smaller Pieces

### Why do we need Chunking?

- LLMs have a **limited context window** — you can't send an entire 100-page document
- Embedding models work better with **short to medium-length text**
- Good chunks help make **retrieval more accurate**

| Method | Principle | Advantages | Disadvantages |
|------|---------|-------|--------|
| **Fixed-size** | Split by character count | Simple, fast | May split in the middle of a sentence |
| **Recursive** ⭐ | Split by nested separators | Better results, preserves structure | Need to define separators |

In [ ]:
%%time
# Load text from files
all_text = ''
for fname in sorted(os.listdir('data')):
    fpath = f'data/{fname}'
    if os.path.isfile(fpath) and fname.endswith('.txt'):
        with open(fpath, 'r', encoding='utf-8') as f:
            all_text += '\n\n' + f.read()

print(f'📄 Total text: {len(all_text)} characters')

In [ ]:
%%time
# ─── Fixed-size Chunking ───
def fixed_chunk(text, size=200, overlap=30):
    chunks = []
    for i in range(0, len(text), size - overlap):
        chunks.append(text[i:i+size])
    return chunks

fixed_chunks = fixed_chunk(all_text, size=200, overlap=30)
print(f'📊 Fixed-size: got {len(fixed_chunks)} chunks')
print(f'\n--- Chunk 1 (notice: it may cut in the middle of a word) ---')
print(fixed_chunks[0][:150] + '...')

In [ ]:
%%time
# ─── Recursive Chunking ⭐ (Recommended) ───
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
    separators=['\n\n', '\n', ' ', '']  # Order: paragraph → line → word → character
)

chunks = splitter.split_text(all_text)
print(f'📊 Recursive: got {len(chunks)} chunks')
print(f'\n--- Chunk 1 (notice: split by paragraph) ---')
print(chunks[0])
print(f'\n--- Last chunk ---')
print(chunks[-1])

### 💡 Observations
- **Fixed-size** is simple but may cut in the middle of words/sentences
- **Recursive** tries splitting by `\n\n` first, and if still too large, by `\n` → much better
- Smaller `chunk_size` → more precise small pieces, but may lose context
- Larger `chunk_size` → more context, but may include noise

> 🎯 In this workshop, we will continue using **Recursive Chunking**

---
## 🔢 Section 2: Embedding — Convert Text into Vectors

### What is an Embedding?

Convert text → into a set of numbers (a vector) that represents meaning

```
"cat" → [0.21, -0.85, 0.44, ...] (1024 dimensions)
"dog" → [0.19, -0.82, 0.41, ...]  ← close to "cat" (both are pets)
"car" → [-0.55, 0.31, -0.12, ...] ← far from "cat" (unrelated)
```

**Model used:** `intfloat/multilingual-e5-large` — supports Thai

> ⚠️ You must add the prefix: `'query: '` for questions, and `'passage: '` for content

In [ ]:
%%time
# Load Embedding Model (first time will download for ~2 minutes)
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
print(f'✅ Model loaded successfully | Vector size: {embed_model.get_sentence_embedding_dimension()}')

In [ ]:
%%time
# Create embeddings for all chunks
passages = ['passage: ' + c for c in chunks]
chunk_embeddings = embed_model.encode(passages, show_progress_bar=True)

print(f'✅ Successfully embedded {len(chunks)} chunks')
print(f'📐 Shape: {chunk_embeddings.shape}')

In [ ]:
%%time
# Try Cosine Similarity
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

query = 'query: AI systems that help answer student questions'
query_emb = embed_model.encode(query)

# Compute similarity with all chunks
scores = cosine_similarity([query_emb], chunk_embeddings)[0]
top_idx = np.argsort(scores)[::-1][:3]

print(f'🔍 Query: "{query}"')
print(f'\n📊 Top-3 most similar chunks:')
for rank, idx in enumerate(top_idx, 1):
    print(f'  {rank}. [score={scores[idx]:.4f}] {chunks[idx][:80]}...')

### 💡 Observations
- Texts with **similar meaning** → have **high cosine similarity** (close to 1.0)
- You must use the prefixes `query:` / `passage:` as required by the model
- Embeddings understand **meaning**, not just exact keywords

> 🎯 Next, we will store these embeddings in a **Vector Database** so we can search faster

---
## 🗄️ Section 3: Qdrant — Vector Database + Retrieve

### Why not use a regular DB?

- Regular DBs (SQL): search by keyword / exact match
- VectorDB: search by **meaning** (semantic search)

**Qdrant** = an easy-to-use, fast VectorDB that supports in-memory mode

In [ ]:
%%time
# Create Qdrant Client + Collection
from qdrant_client import QdrantClient, models

qdrant = QdrantClient(':memory:')  # Use RAM (no server installation needed)

COLLECTION = 'workshop_docs'
VECTOR_SIZE = chunk_embeddings.shape[1]  # 1024

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE)
)

print(f'✅ Created collection "{COLLECTION}" successfully (vector size={VECTOR_SIZE})')

In [ ]:
%%time
# Upsert all chunks into Qdrant
points = [
    models.PointStruct(
        id=i,
        vector=chunk_embeddings[i].tolist(),
        payload={'text': chunks[i], 'chunk_id': i}
    )
    for i in range(len(chunks))
]

qdrant.upsert(collection_name=COLLECTION, points=points)
print(f'✅ Successfully upserted {len(points)} chunks into Qdrant!')

In [ ]:
%%time
# ─── Search from Qdrant ───
def search_qdrant(query_text, top_k=3):
    """Search chunks from Qdrant using a query"""
    q_emb = embed_model.encode(f'query: {query_text}')
    results = qdrant.query_points(
        collection_name=COLLECTION,
        query=q_emb.tolist(),
        limit=top_k
    ).points
    return results

# Try searching
results = search_qdrant('Mahidol AI Center artificial intelligence research')
print('🔍 Query: "Mahidol AI Center artificial intelligence research"')
print(f'\n📊 Top-{len(results)} results from Qdrant:')
for i, r in enumerate(results, 1):
    print(f'  {i}. [score={r.score:.4f}] {r.payload["text"][:80]}...')

### 💡 Observations
- Qdrant searches using **vector similarity** → it understands meaning, not just keywords
- Ask about "medical" → it returns results related to case_healthcare.txt
- `top_k` controls how many results are sent to the LLM

> 🎯 Now we have a complete **RAG Pipeline!** Next, we will build an **Agent** that uses this pipeline

---
### ☕ 10-Minute Break

---

## 📢 Part 2: Agent + Agentic RAG

---
## 🤖 Section 4: Your First Agent — Google ADK

### Agent vs Chatbot

🤔 **Imagine this:** You want to book a flight
- **Chatbot**: "Please choose 1. View flights 2. Check prices 3. Contact staff" ← can only follow a script
- **Agent**: "Find the cheapest flight from Bangkok to Chiang Mai on Friday and book it" ← **thinks + uses tools on its own**

| | Chatbot | Agent |
|---|---------|-------|
| How it works | Q&A by script | **Think → Decide → Use tools** |
| Flexibility | Low (must be hard-coded) | High (adapts) |
| Example | FAQ bot, phone menu | AI booking assistant, AI search assistant, Agentic RAG |

**Google ADK** = Agent Development Kit for building Agents in Python with just a few lines

In [ ]:
%%time
# Create the first Agent
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types

# Create an Agent that acts as a "teaching assistant"
teacher_agent = LlmAgent(
    name='teacher_assistant',
    model='gemini-2.5-flash',
    instruction='You are a university teaching assistant. Answer AI-related questions briefly and clearly in Thai, in no more than 3 sentences.'
)

print(f'✅ Created Agent "{teacher_agent.name}" successfully')

In [ ]:
# ─── Chat with Agent ───
import asyncio

async def chat_with_agent(agent, message):
    """Send a message to the Agent and get a response"""
    runner = InMemoryRunner(agent=agent, app_name='workshop')
    session = await runner.session_service.create_session(
        app_name='workshop', user_id='student'
    )
    
    content = genai_types.Content(
        role='user',
        parts=[genai_types.Part(text=message)]
    )
    
    response_text = ''
    async for event in runner.run_async(
        user_id='student', session_id=session.id, new_message=content
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text
    
    return response_text

# Ask a question
answer = await chat_with_agent(teacher_agent, 'What is RAG?')
print(f'❓ Question: What is RAG?')
print(f'🤖 Agent: {answer}')

### 💡 Observations
- The Agent responds according to the `instruction` we defined
- But the Agent is still answering only from the **LLM's general knowledge**
- It has not searched from **our data** yet → we need to give it a "Tool"

> 🎯 Next, we will create a Tool for the Agent to use

---
## 🔧 Section 5: Tool — Give the Agent a Tool to Use

### What is FunctionTool?

Write a Python function → the Agent can **call it on its own** when needed

> ⚠️ **The docstring is very important!** The Agent reads the docstring to decide which tool to call

In [ ]:
# Create a Custom Tool
from google.adk.tools import FunctionTool

def calculate_bmi(weight_kg: float, height_cm: float) -> str:
    """Calculate BMI from weight (kilograms) and height (centimeters)
    
    Args:
        weight_kg: weight in kilograms
        height_cm: height in centimeters
    """
    height_m = height_cm / 100
    bmi = weight_kg / (height_m ** 2)
    if bmi < 18.5: status = 'Underweight'
    elif bmi < 25: status = 'Normal weight'
    elif bmi < 30: status = 'Overweight'
    else: status = 'Obese'
    return f'BMI = {bmi:.1f} ({status})'

bmi_tool = FunctionTool(calculate_bmi)

# Create an Agent with a Tool
health_agent = LlmAgent(
    name='health_assistant',
    model='gemini-2.5-flash',
    instruction='You are a health assistant. Respond in Thai. Use the calculate_bmi tool when the user provides weight and height.',
    tools=[bmi_tool]
)

print('✅ Created Health Agent + BMI Tool successfully')

In [ ]:
# Try Agent + Tool
answer = await chat_with_agent(health_agent, 'I weigh 70 kg and am 175 cm tall. What is my BMI?')
print(f'❓ Question: I weigh 70 kg and am 175 cm tall.')
print(f'🤖 Agent: {answer}')

### 💡 Observations
- The Agent **reads the docstring** and decides to call `calculate_bmi` on its own
- No need to hard-code it: if asked about something else, the Agent will not call the tool
- **Clear docstrings = correct tool usage by the Agent**

> 🎯 Next, we will build a **RAG Tool** that searches from Qdrant!

---
## 🔍 Section 6: RAG Agent ⭐ — Agent + VectorDB

### The heart of this workshop!

Connect the Agent + Qdrant Pipeline from Part 1

```
Question → Agent decides → calls search_knowledge
→ searches from Qdrant → gets context back → Agent generates the answer
```

In [ ]:
# Create RAG Tool — search from Qdrant
def search_knowledge(query: str) -> str:
    """Search for information from a knowledge base of AI case studies in Thailand
    Use this when looking for information about AI in different sectors such as healthcare, finance, agriculture, and education.
    
    Args:
        query: the question or topic to search for
    """
    results = search_qdrant(query, top_k=3)
    context = '\n\n'.join([f'[{i+1}] {r.payload["text"]}' for i, r in enumerate(results)])
    return f'Search results: {len(results)} items\n{context}'

rag_tool = FunctionTool(search_knowledge)

# Create RAG Agent
rag_agent = LlmAgent(
    name='rag_assistant',
    model='gemini-2.5-flash',
    instruction="""You are an AI Assistant that answers questions about AI in Thailand

Rules:
1. Always use the search_knowledge tool to search before answering
2. Answer only from the retrieved information; do not make things up
3. If no information is found, say so directly
4. Answer in Thai, briefly and clearly""",
    tools=[rag_tool]
)

print('✅ Created RAG Agent successfully!')

In [ ]:
# ─── Try RAG Agent ───
questions = [
    'How does AI help in hospitals?',
    'What AI projects does the ICT Faculty at Mahidol have?',
    'How do banks use AI?'
]

for q in questions:
    answer = await chat_with_agent(rag_agent, q)
    print(f'\n{"="*60}')
    print(f'❓ {q}')
    print(f'🤖 {answer}')

### 💡 Observations
- The Agent **decides on its own** to search from Qdrant
- The answer comes from **real retrieved data**, not just the LLM's general knowledge
- Different from regular RAG: an **Agent can choose** whether to search or answer directly

> 🎯 This is **Agentic RAG** — next, we will add Memory so it can remember the conversation!

---
## 🚀 Section 7: Agentic RAG + Memory ⭐

### Add Session Memory

Let the Agent **remember the conversation** → so follow-up questions don't need repeated context

In [ ]:
# ─── Agentic RAG + Memory ───
from google.adk.sessions import InMemorySessionService

# Use the same RAG Agent, but add a session so it can remember
runner = InMemoryRunner(agent=rag_agent, app_name='agentic_rag')

async def chat_with_memory(runner, session_id, message):
    content = genai_types.Content(
        role='user',
        parts=[genai_types.Part(text=message)]
    )
    response_text = ''
    async for event in runner.run_async(
        user_id='student', session_id=session_id, new_message=content
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text
    return response_text

# Create session
session = await runner.session_service.create_session(
    app_name='agentic_rag', user_id='student'
)

# Ask 3 follow-up questions
conversations = [
    'What AI applications are used in Thai agriculture?',
    'How is NLP used?',           # Does not specify the domain — Agent must remember we are discussing agriculture
    'What are the challenges?'    # Continues from the same topic
]

for q in conversations:
    answer = await chat_with_memory(runner, session.id, q)
    print(f'\n{"="*60}')
    print(f'❓ {q}')
    print(f'🤖 {answer}')

### 💡 Observations
- In questions 2 and 3, the topic is **not explicitly stated**, but the Agent remembers that the discussion is about agriculture
- This is the power of **Session Memory** — it allows the Agent to hold a continuous conversation
- **Agentic RAG = Agent + RAG + Memory** → a system that can search + think + remember

> 🎯 If we combine everything: Data Pipeline + Agent + Tool + RAG + Memory = a **full Agentic RAG system!**

---
## 📋 Summary: What We Learned Today

| Part | Section | What we did |
|------|---------|--------|
| 1 | Chunking | Split text with Recursive Splitter |
| 1 | Embedding | Convert to vectors (multilingual-e5-large) |
| 1 | Qdrant | Store + search vectors |
| 2 | Agent | Build an Agent with Google ADK |
| 2 | Tool | Let the Agent use FunctionTool |
| 2 | RAG Agent ⭐ | Agent searches from VectorDB |
| 2 | Agentic RAG ⭐ | Agent + RAG + Memory |

### 🔑 Key Takeaways

1. **Data quality matters most** — "Garbage in, garbage out"
2. **Agent ≠ Chatbot** — an Agent can think + use tools
3. **Agentic RAG** = an Agent that decides for itself whether to search or answer directly

---

### 🧪 Exercise (1 hour)

Open **`agentic_rag_4hr_homework.ipynb`** and follow the instructions (10 points)

> _"The best way to learn AI is to build with AI."_